# Exp 04: Oversampling VIP animals

Card: [exp04_oversampling_vip.md](../../experiments/exp04_oversampling_vip.md)


In [ ]:
import os

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd

from core.config import ROOT
from core.cv import VIP, CVResult, Config, CrossValidator
from core.gradcam import GradCamFigures
from core.tracking import Experiment

SMOKE = os.environ.get("SMOKE") == "1"
if SMOKE:
    OVERRIDES = {
        "experiment": "cnn-model-oversampling-vip-smoke",
        "epochs": 1,
        "img_size": 64,
        "n_other": 20,
        "num_workers": 0,
        "retrain": True,
        "log_every_n_steps": 1,
    }
    N_SHOW = 2  # images per Grad-CAM figure
else:
    OVERRIDES = {}
    N_SHOW = 4
CONFIG_DIR = ROOT / "sections" / "model" / "configs"
CONFIG = Config.from_toml(
    CONFIG_DIR / "exp04_oversampling_vip" / "resnet18.toml", **OVERRIDES
)
PLAIN = Config.from_toml(
    CONFIG_DIR / "exp01_lecture_models" / "resnet18.toml"
)  # trained in exp01, only read
FIGURES = ROOT / "sections" / "model" / "figures"
SAVE_FIGURES = not SMOKE

# dataviz reference palette, light mode; categorical slots in fixed order
MUTED = "#898781"
SERIES = {"plain (exp01)": "#eb6834", "oversampling x8": "#1baf7a"}
plt.rcParams.update(
    {
        "figure.facecolor": "#fcfcfb",
        "axes.facecolor": "#fcfcfb",
        "savefig.facecolor": "#fcfcfb",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.edgecolor": "#52514e",
        "xtick.color": "#52514e",
        "ytick.color": "#52514e",
        "axes.titlesize": "medium",
        "axes.titlecolor": "#52514e",
        "axes.grid": True,
        "grid.color": "#e1e0d9",
        "legend.frameon": False,
        "legend.fontsize": "small",
    }
)


## Training


In [ ]:
results = {
    "plain (exp01)": CVResult.load(PLAIN),
    "oversampling x8": CrossValidator(CONFIG).run(),
}

## Training curves


In [ ]:
panels = [
    ("Loss (weighted BCE)", "train_loss_epoch", "val_loss"),
    ("mAP", "train_map", "val_map"),
]
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (title, train_key, val_key) in zip(axes, panels):
    for name, r in results.items():
        epochs = np.arange(1, r.config.epochs + 1)
        ax.plot(epochs, r.mean_curve(train_key), color=SERIES[name], ls=":")
        ax.plot(epochs, r.mean_curve(val_key), color=SERIES[name], label=name)
    ax.set(title=f"{title}, mean over {len(CONFIG.folds)} folds", xlabel="epoch")
axes[1].plot([], [], color=MUTED, label="validation")
axes[1].plot([], [], color=MUTED, ls=":", label="train")
axes[1].legend(loc="lower right", title="sampling", alignment="left")
fig.tight_layout()
figures = {"exp04_oversampling_vip_training.png": fig}
plt.show()

## AP comparison


In [ ]:
fold_ap_df = pd.concat(
    {name: r.fold_ap for name, r in results.items()}, names=["sampling"]
)
ap = fold_ap_df.groupby("sampling", sort=False).mean()
ap.round(3)

In [ ]:
# difference to exp01 counts only if it has same sign on both folds
diff = (
    results["oversampling x8"].fold_ap - results["plain (exp01)"].fold_ap
).T.add_prefix("diff fold ")
diff["same sign on both folds"] = (diff > 0).all(axis=1) | (diff < 0).all(axis=1)
diff.round(3)

## Grad-CAM


In [ ]:
cams = GradCamFigures(CONFIG)
for (cls, outcome), fig in cams.figures(VIP, N_SHOW).items():
    figures[f"exp04_oversampling_vip_grad_cam_{cls.lower()}_{outcome}.png"] = fig
plt.show()

In [ ]:
with Experiment(CONFIG.experiment).run("summary", {"val_mode": "fast"}) as run:
    mlflow.log_table(fold_ap_df.reset_index(), "fold_ap.json")
    mlflow.log_table(diff.reset_index(names="metric"), "diff.json")
    mlflow.log_metrics({f"threshold_{c}": cams.thresholds[c] for c in VIP})
    if SAVE_FIGURES:
        run.log_figures(figures, FIGURES)